[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajitpanday80/ai-learn/blob/main/notebooks/llm-how-work.ipynb)

# 🧠 How Do LLMs Work?

**Section:** LLMs

A Large Language Model (LLM) is a neural network trained on one simple task: **predict the next token**. When you repeat that step many times, the model produces a whole paragraph of text.

The pipeline looks like this:

1. **Tokenization**: text is split into tokens (whole words or pieces of words), and each token becomes an integer ID.
2. **Embedding + Transformer**: the IDs are turned into vectors. Stacked attention layers then mix information across the whole context.
3. **Logits → Probabilities**: the model gives a score for *every* token in its vocabulary. Softmax turns those scores into a probability distribution.
4. **Sampling**: we pick the next token, add it to the input, and repeat. This is called *autoregressive generation*.

In this notebook you'll open up **GPT-2** (124M parameters) and watch each step happen.

> ⚡ **GPU tip:** In Colab, go to **Runtime > Change runtime type > GPU** to make it faster. GPT-2 small also runs fine on CPU.

In [ ]:
!pip install -q transformers torch matplotlib

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

plt.style.use('dark_background')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded GPT-2 with {n_params / 1e6:.0f}M parameters')
print(f'Vocabulary size: {tokenizer.vocab_size:,} tokens')
print(f'Context window: {model.config.n_positions} tokens')
print(f'Transformer layers: {model.config.n_layer}, attention heads: {model.config.n_head}')

## 1️⃣ Step 1: Tokenization

LLMs don't read characters or words. They read **tokens**. GPT-2 uses *Byte-Pair Encoding (BPE)*:
- Common words like `the` or ` cat` are a single token.
- Rare words are split into sub-word pieces.
- A leading space is usually part of the token (` cat` and `cat` are different tokens!).

🔧 **Experiment:** Change `text` and the `words` list below. Try your name, code, emoji, numbers, or another language. How many tokens does each one use?

In [ ]:
# 🔧 Try your own sentence!
text = 'Large language models predict the next token, one step at a time.'

ids = tokenizer.encode(text)
tokens = [tokenizer.decode([i]) for i in ids]

print(f'{len(text)} characters -> {len(ids)} tokens\n')
print(f'{"ID":>6}  token')
for i, t in zip(ids, tokens):
    print(f'{i:>6}  |{t}|')

# 🔧 Compare how different words get split
words = ['cat', ' cat', 'unbelievably', 'Pneumonoultramicroscopic', '12345', 'ChatGPT', 'def foo():']
counts = []
print('\nSub-word splits:')
for w in words:
    pieces = [tokenizer.decode([i]) for i in tokenizer.encode(w)]
    counts.append(len(pieces))
    print(f'{w!r:>28} -> {pieces}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh([repr(w) for w in words][::-1], counts[::-1], color='#4cc9f0')
ax.set_xlabel('Number of tokens')
ax.set_title('How many tokens does each string cost?')
plt.tight_layout()
plt.show()

## 2️⃣ Step 2: Next-Token Prediction

The token IDs go through 12 transformer layers. At the end, the model outputs **logits**: one raw score for each of the ~50,000 tokens in the vocabulary, at every position.

We only need the scores at the **last position**, because that is the model's guess for what comes next. Applying **softmax** turns them into probabilities that add up to 1:

$$P(\text{token}_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

🔧 **Experiment:** Try prompts where the answer is obvious (`The capital of France is`) and prompts where it's open-ended (`My favorite food is`). Compare how *peaked* or *flat* the distribution looks.

In [ ]:
# 🔧 Change the prompt and see what the model expects next
prompt = 'The capital of France is'
top_k = 10

inputs = tokenizer(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    logits = model(**inputs).logits  # shape: (batch, seq_len, vocab_size)

print('Logits shape:', tuple(logits.shape), '-> (batch, tokens in prompt, vocabulary)')

next_logits = logits[0, -1]               # scores for the NEXT token
probs = torch.softmax(next_logits, dim=-1)  # turn scores into probabilities
top = torch.topk(probs, top_k)

labels = [repr(tokenizer.decode([i])) for i in top.indices.tolist()]
vals = top.values.cpu().numpy()

print(f'\nTop {top_k} next tokens:')
for label, v in zip(labels, vals):
    print(f'{label:>15}  {v:6.2%}')

entropy = -(probs * torch.log(probs + 1e-12)).sum().item()
print(f'\nEntropy of the distribution: {entropy:.2f} nats (lower = more confident)')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(labels[::-1], vals[::-1], color='#f72585')
ax.set_xlabel('Probability')
ax.set_title('Next-token predictions after: ' + prompt + ' ...')
plt.tight_layout()
plt.show()

## 3️⃣ Step 3: Autoregressive Generation and Temperature

To produce text, we run a loop: **predict → pick a token → append it → repeat**. How we *pick* the token makes a big difference:

| Strategy | How it works | Effect |
|---|---|---|
| **Greedy** (`temperature=0`) | Always choose the most likely token | Predictable, often repetitive |
| **Temperature** | Divide logits by `T` before softmax | `T<1` makes the model more confident, `T>1` makes it more random |
| **Top-k** | Sample only from the `k` most likely tokens | Removes unlikely, strange tokens |

🔧 **Experiments to try:**
- Change `prompt`, `temperatures`, `top_k`, and `seed` below.
- Set `temperature=0` and generate 60 tokens. Do you see the text start to repeat itself?
- Try `temperature=3.0`. When does the output turn into nonsense?
- Look at the plots: how does temperature change the *shape* of the distribution?

💡 **Key insight:** ChatGPT, Claude, and other chat models run this *same loop*. They are simply much bigger and have extra training (instruction tuning and RLHF) so that the next tokens form helpful answers.

In [ ]:
def generate(prompt, max_new_tokens=40, temperature=1.0, top_k=50, seed=0):
    torch.manual_seed(seed)
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        if temperature == 0:
            next_id = torch.argmax(logits).view(1, 1)          # greedy
        else:
            logits = logits / temperature                        # temperature scaling
            if top_k:
                kth = torch.topk(logits, top_k).values[-1]
                logits[logits < kth] = -float('inf')             # top-k filtering
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1).view(1, 1)     # sample
        ids = torch.cat([ids, next_id], dim=1)                  # append and repeat
    return tokenizer.decode(ids[0])

# 🔧 Experiment here!
prompt = 'Once upon a time, in a city run by robots,'
temperatures = [0, 0.7, 1.5]
top_k = 50
seed = 42

for t in temperatures:
    print(f'=== temperature = {t} ===')
    print(generate(prompt, temperature=t, top_k=top_k, seed=seed))
    print()

# Visualize how temperature reshapes the next-token distribution
with torch.no_grad():
    base_logits = model(**tokenizer(prompt, return_tensors='pt').to(device)).logits[0, -1]

plot_temps = [0.5, 1.0, 2.0]
n_show = 15
fig, axes = plt.subplots(1, len(plot_temps), figsize=(16, 4.5), sharey=True)
colors = ['#4cc9f0', '#b5179e', '#f8961e']
for ax, t, c in zip(axes, plot_temps, colors):
    p = torch.softmax(base_logits / t, dim=-1)
    top = torch.topk(p, n_show)
    ax.bar(range(n_show), top.values.cpu().numpy(), color=c)
    ax.set_xticks(range(n_show))
    ax.set_xticklabels([repr(tokenizer.decode([i])) for i in top.indices.tolist()], rotation=70, fontsize=8)
    ax.set_title(f'Temperature = {t}  (top-1: {top.values[0].item():.1%})')
axes[0].set_ylabel('Probability')
fig.suptitle('Same logits, different temperatures', fontsize=14)
plt.tight_layout()
plt.show()